# Tutorial 03: Complete Active Learning

This tutorial puts together the active learning loop of tutorial 01 and the diffusion model of tutorial 02, with a
DFT oracle. In tutorial 01, the oracle labels the full 512-atom structure where the molecular dynamics stopped: this
is fine with Stillinger-Weber, but far too expensive with DFT. Here, the Generate stage excises the environment of
each uncertain atom, and repaints a 64-atom cell around it with the diffusion model. DFT then only labels small
cells, which still contain the atomic environment where the MTP was uncertain.

This tutorial follows `examples/05_activelearning_mtp_md_abinit_repaint.py`. The components are:

- **MLIP:** a Moment Tensor Potential (MTP), fitted by MLIP-3, as in tutorial 01.
- **Dynamic driver:** molecular dynamics, run by LAMMPS, as in tutorial 01.
- **Sample maker:** excise-and-repaint, with the pretrained diffusion model of tutorial 02.
- **Oracle:** Abinit (DFT).

The loop, its stages and the files of a campaign are described in tutorial 01; this tutorial focuses on what is new:
the excise-and-repaint sample maker and the Abinit oracle.

> **Note**
>
> This tutorial calls three external programs: a LAMMPS with the `mtp/extrapolation` pair style
> ([lammps-mtp-kokkos](https://github.com/RichardZJM/lammps-mtp-kokkos)), the `mlp` executable of
> [MLIP-3](https://gitlab.com/ashapeev/mlip-3), and [Abinit](https://www.abinit.org).
>
> Before launching Jupyter, define the following environment variables in your terminal (or add them to your
> `~/.bashrc` or `~/.zshrc`):
>
> ```bash
> export DFMSMD_LAMMPS_PATH=Replace_with_absolute_path_to_lmp  # Change this line
> export DFMSMD_MLP_PATH=Replace_with_absolute_path_to_mlp     # Change this line
> export DFMSMD_ABINIT_MPI_TASKS=4  # Optional, 4 by default; the number of physical cores of your machine
> export DFMSMD_ABINIT_COMMAND="env OMP_NUM_THREADS=1 OPENBLAS_NUM_THREADS=1 abinit"  # Optional, this is the default
> ```
>
> Then launch Jupyter from that same terminal: a notebook only sees the environment variables of the shell that
> started Jupyter. The silicon pseudopotential and the pretrained diffusion model are in
> `examples/references_files`.

In [ ]:
import warnings

# To suppress warnings for clearer output
warnings.simplefilter("ignore")

In [ ]:
# Define the directory where artifacts will be written. Delete it if it exists to start clean.
import shutil
from diffusion_for_multi_scale_molecular_dynamics import TOP_DIR

output_path = TOP_DIR / "tutorials" / "output" / "03-complete_active_learning"

# Restart from an empty directory
if output_path.is_dir():
    shutil.rmtree(output_path)

output_path.mkdir(parents=True, exist_ok=True)

# Stage 1, driver: Molecular Dynamics

As in tutorial 01, the `MdDriver` runs with LAMMPS a molecular dynamics of a 512-atom diamond silicon supercell (NVT
at 300 K, 1 fs time step, 1000 steps), which stops as soon as the MTP is uncertain about an atom.

In [ ]:
import os
from pathlib import Path

from ase.build import bulk

from diffusion_for_multi_scale_molecular_dynamics.dynamic_driver.md_driver.md_driver import MdDriver
from diffusion_for_multi_scale_molecular_dynamics.oracle.lammps_runner import SubprocessLammpsRunner

lammps_runner = SubprocessLammpsRunner(lammps_executable_path=Path(os.environ["DFMSMD_LAMMPS_PATH"]),
                                       mpi_processors=1,
                                       openmp_threads=1,
                                       mpi_executable="mpirun")

lattice_constant = 5.43  # Å, diamond silicon

# The (orthogonal) starting configuration of the dynamic driver: 512 atoms of diamond silicon.
initial_configuration = bulk("Si", "diamond", a=lattice_constant, cubic=True).repeat((4, 4, 4))

dynamic_driver = MdDriver(lammps_runner=lammps_runner,
                          initial_configuration=initial_configuration,
                          temperature=300.0,
                          timestep=0.001,
                          number_of_steps=1000)

# Stage 2, generate: Excise and Repaint

The `ExciseAndRepaintSampleMaker` turns the uncertain 512-atom structure into small cells ready to be labelled by
DFT. For each uncertain atom:

1. the `ThresholdAtomSelector` picks the atoms whose extrapolation grade γ is above 2;
2. the `SphericalExcision` excises every atom within 4.2 Å of the uncertain atom: the atom itself, its 4 first
   neighbors and its 12 second neighbors, 17 atoms in total;
3. these atoms are placed at the center of a 64-atom cubic cell of 10.86 Å (twice the cubic cell of silicon), larger
   than the 8.4 Å diameter of the excised sphere;
4. the `ConstrainedLangevinGenerator` keeps them fixed, and repaints the other 47 atoms of the cell with the pretrained
   diffusion model of tutorial 02, with the noise schedule of its training.

At most 4 uncertain atoms are excised per round (`max_constrained_substructure`), with one repainted cell each
(`number_of_samples_per_substructure`).

- `epoch_N/generate.traj`: the repainted cells of round N, not labelled yet. In each cell, the first atoms are the
  excised ones, the central (uncertain) atom first.

In [ ]:
from diffusion_for_multi_scale_molecular_dynamics.diffusion_model.models.axl_diffusion_lightning_model import \
    AXLDiffusionLightningModel
from diffusion_for_multi_scale_molecular_dynamics.diffusion_model.noise_schedulers.noise_parameters import \
    NoiseParameters
from diffusion_for_multi_scale_molecular_dynamics.sample_maker.atom_selector.threshold_atom_selector import (
    ThresholdAtomSelector, ThresholdAtomSelectorParameters)
from diffusion_for_multi_scale_molecular_dynamics.sample_maker.excise_and_repaint_sample_maker import (
    ExciseAndRepaintSampleMaker, ExciseAndRepaintSampleMakerArguments)
from diffusion_for_multi_scale_molecular_dynamics.sample_maker.excisor.spherical_excisor import (
    SphericalExcision, SphericalExcisionArguments)
from diffusion_for_multi_scale_molecular_dynamics.sample_maker.generator.predictor_corrector_axl_generator import \
    PredictorCorrectorSamplingParameters

references_files = TOP_DIR / "examples" / "references_files"

# Å, between the second (3.84 Å) and third (4.50 Å) neighbors of silicon; also the cutoff of the MTP (see Stage 4).
radial_cutoff = 4.2

# The repainted cell: 64 atoms, twice the cubic cell of silicon.
sample_box_size = [2 * lattice_constant] * 3

atom_selector = ThresholdAtomSelector(ThresholdAtomSelectorParameters(uncertainty_threshold=2.0))
environment_excisor = SphericalExcision(SphericalExcisionArguments(radial_cutoff=radial_cutoff))
sample_maker_arguments = ExciseAndRepaintSampleMakerArguments(element_list=["Si"],
                                                              sample_box_size=sample_box_size,
                                                              max_constrained_substructure=4,
                                                              number_of_samples_per_substructure=1)

diffusion_model_path = references_files / "pretrainedcartesianmodel_Si64_64-4_epoch43.ckpt"
diffusion_model = AXLDiffusionLightningModel.load_from_checkpoint(diffusion_model_path, map_location="cpu").axl_network
diffusion_model.eval()

# The noise schedule of the training of the pretrained model (see tutorial 02).
sigma_min = 1.088740e-3
noise_parameters = NoiseParameters(total_time_steps=1000,
                                   schedule_type="linear",
                                   sigma_min_cart=sigma_min,
                                   sigma_max_cart=2.177481,
                                   corrector_step_epsilon=1.6 * sigma_min**2)
sampling_parameters = PredictorCorrectorSamplingParameters(num_atom_types=1,
                                                           number_of_atoms=64,
                                                           number_of_samples=1,
                                                           use_fixed_lattice_parameters=True,
                                                           cell_dimensions=sample_box_size,
                                                           number_of_corrector_steps=2,
                                                           progress_bar=False)

sample_maker = ExciseAndRepaintSampleMaker(sample_maker_arguments=sample_maker_arguments,
                                           atom_selector=atom_selector,
                                           environment_excisor=environment_excisor,
                                           noise_parameters=noise_parameters,
                                           sampling_parameters=sampling_parameters,
                                           diffusion_model=diffusion_model)

# Stage 3, oracle: Abinit

The `AbinitSinglePointCalculator` labels each repainted cell with its DFT energy and forces. The parameters follow the
ASE unit convention (eV, Å), and are cheap rather than converged, to keep the tutorial short:

- `ecut=550.0`: the plane-wave cutoff, 550 eV (about 20 Ha).
- `ngkpt=[1, 1, 1]`, `nshiftk=1`, `shiftk=[0.0, 0.0, 0.0]`: the Γ point only, enough for a qualitative 64-atom cell.
- `toldfe=1.0e-6`, `nstep=100`: the self-consistent cycle stops when the energy changes by less than 10<sup>-6</sup> Ha,
  or after 100 steps.
- `occopt=3`, `tsmear=0.1`: a Fermi-Dirac smearing of 0.1 eV, since disordered repainted cells can be metallic.
- `paral_kgb=1`, `autoparal=1`: with a single k-point, Abinit splits the bands and the FFT over the MPI tasks itself.
- `prtwf=0`, `prtden=0`: no wavefunction or density files, whose MPI-IO write can hang.
- `nsym=1`: no symmetry, since the repainted cells have none, and the perfect 64-atom crystal has more symmetries than
  Abinit allows.

Abinit runs with `mpirun` on `DFMSMD_ABINIT_MPI_TASKS` tasks (4 by default), one OpenMP thread each (see
`DFMSMD_ABINIT_COMMAND`).

Before the campaign, the oracle also labels the starting training data of the MTP: a 64-atom cell of diamond silicon,
the same cell as the repainted samples. Its DFT energy is the reference of the analysis below.

- `epoch_N/oracle.traj`: the repainted cells of round N with their DFT energy and forces, added to the training data.
- `epoch_N/oracle/`: the working directory of Abinit for round N.

In [ ]:
from diffusion_for_multi_scale_molecular_dynamics.oracle.abinit_runner import AbinitRunner
from diffusion_for_multi_scale_molecular_dynamics.oracle.abinit_single_point_calculator import \
    AbinitSinglePointCalculator
from diffusion_for_multi_scale_molecular_dynamics.utils.structure_utils import label_configurations

number_of_mpi_tasks = int(os.environ.get("DFMSMD_ABINIT_MPI_TASKS", 4))
abinit_command = os.environ.get("DFMSMD_ABINIT_COMMAND", "env OMP_NUM_THREADS=1 OPENBLAS_NUM_THREADS=1 abinit")
abinit_runner = AbinitRunner(abinit_command=abinit_command, mpi_runner="mpirun", number_of_mpi_tasks=number_of_mpi_tasks)

parameters = dict(ecut=550.0, ngkpt=[1, 1, 1], nshiftk=1, shiftk=[0.0, 0.0, 0.0], toldfe=1.0e-6, nstep=100,
                  occopt=3, tsmear=0.1, paral_kgb=1, autoparal=1, prtwf=0, prtden=0, nsym=1)
oracle = AbinitSinglePointCalculator(parameters=parameters,
                                     pseudopotentials={"Si": references_files / "Si.psp8"},
                                     abinit_runner=abinit_runner)

# The starting training data of the MTP: a 64-atom cell of diamond silicon, labelled by the oracle.
crystal = bulk("Si", "diamond", a=lattice_constant, cubic=True).repeat((2, 2, 2))
provided_configurations = label_configurations([crystal], oracle)

print(f"Energy: {provided_configurations[0].get_potential_energy() / len(crystal):.4f} eV/atom")
print(f"Largest force component: {abs(provided_configurations[0].get_forces()).max():.2e} eV/Å")

# Stage 4, train: Moment Tensor Potential

As in tutorial 01, the `MtpMlip` is fitted by MLIP-3 on the energies and forces of every labelled structure, here with
an MTP of level 8. Its cutoff `max_dist` is the radius of the excision, 4.2 Å: the excised sphere then holds the whole
environment the MTP sees around the uncertain atom, so that the DFT label of the repainted cell describes exactly the
environment where the MTP was uncertain.

If the provided configurations do not hold enough atomic environments for the active learning set of the MTP,
perturbed copies are labelled by the oracle (see tutorial 01): with Abinit, these are DFT calculations too, before the
first round.

In [ ]:
from diffusion_for_multi_scale_molecular_dynamics.mlip.mtp.mtp_configuration import MtpConfiguration
from diffusion_for_multi_scale_molecular_dynamics.mlip.mtp.mtp_mlip import MtpMlip
from diffusion_for_multi_scale_molecular_dynamics.mlip.mtp.mtp_trainer import MtpTrainer

mtp_configuration = MtpConfiguration(elements=["Si"], level=8, max_dist=radial_cutoff,
                                     energy_weight=1.0, force_weight=0.01, stress_weight=0.0, site_en_weight=0.0)
mtp_trainer = MtpTrainer(mtp_configuration=mtp_configuration, mlp_executable_path=Path(os.environ["DFMSMD_MLP_PATH"]))
mlip = MtpMlip(mtp_trainer=mtp_trainer, lammps_runner=lammps_runner)

# Active Learning

As in tutorial 01, `ActiveLearning` puts the four components together, and `run_campaign` runs the loop until the
molecular dynamics completes without meeting an uncertain atom. With `restart_from_stage="auto"`, running this cell
again resumes just after the last stage that completed, which avoids redoing DFT calculations.

The campaign should last about 20 epochs and took about 30 minutes to run on a macbook pro with 18 cores.

In [ ]:
from diffusion_for_multi_scale_molecular_dynamics.active_learning_loop.active_learning import ActiveLearning

active_learning = ActiveLearning(oracle_single_point_calculator=oracle,
                                 sample_maker=sample_maker,
                                 dynamic_driver=dynamic_driver,
                                 mlip=mlip)

active_learning.run_campaign(uncertainty_threshold=2.0,  # MTP extrapolation grade (gamma) that stops the dynamic driver
                             working_directory=output_path,
                             provided_configurations=provided_configurations,
                             maximum_number_of_rounds=2,
                             restart_from_stage="auto")  # either {"auto", "driver", "generate", "oracle", "train"}

# Analysis

Once the campaign is complete, we can analyze the run.

**Molecular dynamics steps per epoch.** The number of steps the molecular dynamics ran at each epoch before an
uncertain atom stopped it. As the MTP learns, the dynamics should run longer, until it completes the 1000 steps
without interruption. It is an orange flag if the simulation does few steps every epoch, and then completes the
simulation in one long epoch without interruption.

In [ ]:
import re

from matplotlib import pyplot as plt
from matplotlib.ticker import MaxNLocator

epoch_directories = sorted(output_path.glob("epoch_*"), key=lambda path: int(path.name.removeprefix("epoch_")))
epochs = [int(path.name.removeprefix("epoch_")) for path in epoch_directories]

# LAMMPS ends log.lammps with the number of steps it ran: "Loop time of ... for 637 steps with 512 atoms".
number_of_steps = []
for epoch_directory in epoch_directories:
    lammps_log = (epoch_directory / "dynamic" / "log.lammps").read_text()
    number_of_steps.append(int(re.search(r"Loop time of .* for (\d+) steps", lammps_log).group(1)))

plt.plot(epochs, number_of_steps, "o-")
plt.yscale("log")
plt.gca().xaxis.set_major_locator(MaxNLocator(integer=True))
plt.xlabel("Epoch")
plt.ylabel("Molecular dynamics steps")
plt.show()

**DFT energy of the labelled cells per epoch.** The DFT energy per atom of each repainted cell labelled at each epoch,
above the energy of the 64-atom crystal. The excised sphere comes from the molecular dynamics at 300 K, where a
harmonic crystal holds on average 3/2 k<sub>B</sub>T ≈ 39 meV/atom of potential energy, and the rest of the cell from
the diffusion model, trained on silicon near equilibrium: the labelled cells should lie a few tens of meV/atom above
the crystal. It is a red flag if the energy is far above (hundreds of meV/atom): either the molecular dynamics visits
unphysical configurations, or the repainted atoms do not join the excised sphere, and the training data fills with
defects the dynamics never visits.

In [ ]:
import ase.io

crystal_energy_per_atom = provided_configurations[0].get_potential_energy() / len(provided_configurations[0])

# The labelled cells of each epoch, with their DFT energy, are in epoch_N/oracle.traj. The last epoch completes its
# dynamics: it has no cell to label, hence no oracle.traj.
labelled_epochs, energies_above_crystal = [], []
for epoch, epoch_directory in zip(epochs, epoch_directories):
    if (epoch_directory / "oracle.traj").is_file():
        for atoms in ase.io.read(epoch_directory / "oracle.traj", index=":"):
            labelled_epochs.append(epoch)
            energies_above_crystal.append(1000.0 * (atoms.get_potential_energy() / len(atoms) - crystal_energy_per_atom))

plt.plot(labelled_epochs, energies_above_crystal, "o", label="Labelled cell")
plt.axhline(0.0, color="gray", linestyle="--", label="Crystal at equilibrium")
plt.gca().xaxis.set_major_locator(MaxNLocator(integer=True))
plt.xlabel("Epoch")
plt.ylabel("DFT energy above the crystal (meV/atom)")
plt.legend()
plt.show()

**Quality of the repainted cells per epoch.** For each repainted cell, the fraction of atoms with exactly 4 neighbors
closer than 2.6 Å, as in diamond silicon, and the distance between its two closest atoms. At 300 K, the thermal
vibrations spread the bond length of silicon to 2.36 ± 0.05 Å. The closest pair is the shortest of the 128 bonds of a
64-atom cell: in a 64-atom cell at thermal equilibrium, it lies between 2.21 and 2.28 Å in 98% of the cells, and
below 2.21 Å in only 1% of them.

You should, however, expect the excised sphere to capture an extreme environment. The molecular dynamics of the
512-atom cell goes through up to 1000 time steps of 1024 bonds each, and the MTP stops it at the most unusual
environment it meets, often around an unusually short or long bond. The closest pair of the repainted cells can then fall
below the thermal range of a 64-atom cell, while nearly every atom stays 4-coordinated. It is a red flag if many
atoms are not 4-coordinated, or if the closest pair is well below the thermal range, under 2 Å: the repainted atoms
do not join the excised sphere, or overlap, and DFT labels a defective cell instead of the environment where the MTP
was uncertain.

Many dots on one epoch means that many cells were repainted during this epoch. 

In [ ]:
import numpy as np
from ase.neighborlist import neighbor_list

bond_cutoff = 2.6  # Å, between the first and the second neighbors of diamond silicon
# Å, the range of the closest pair in 98% of the 64-atom cells at 300 K (1% below, 1% above).
closest_pair_range = (2.21, 2.28)

# The repainted cells of each epoch are in epoch_N/generate.traj.
generated_epochs, four_coordinated_fractions, closest_distances = [], [], []
for epoch, epoch_directory in zip(epochs, epoch_directories):
    if (epoch_directory / "generate.traj").is_file():
        for atoms in ase.io.read(epoch_directory / "generate.traj", index=":"):
            coordination_numbers = np.bincount(neighbor_list("i", atoms, bond_cutoff), minlength=len(atoms))
            distances = atoms.get_all_distances(mic=True)
            np.fill_diagonal(distances, np.inf)
            generated_epochs.append(epoch)
            four_coordinated_fractions.append(np.mean(coordination_numbers == 4))
            closest_distances.append(distances.min())

figure, (left_axis, right_axis) = plt.subplots(1, 2, figsize=(12, 4))
left_axis.plot(generated_epochs, four_coordinated_fractions, "o")
left_axis.axhline(1.0, color="gray", linestyle="--", label="Diamond")
left_axis.set_ylim(0.0, 1.05)
left_axis.set_yticks(np.linspace(0.0, 1.0, 6))
left_axis.set_ylabel("Fraction of 4-coordinated atoms")
left_axis.legend()
right_axis.plot(generated_epochs, closest_distances, "o")
right_axis.axhspan(*closest_pair_range, color="gray", alpha=0.2, label="Closest pair at 300 K (98% of cells)")
right_axis.set_ylabel("Closest pair distance (Å)")
right_axis.legend()
for axis in (left_axis, right_axis):
    axis.xaxis.set_major_locator(MaxNLocator(integer=True))
    axis.set_xlabel("Epoch")
plt.show()